In [62]:
# Coinbase Client Test - Ed25519 JWT Authentication
# This notebook tests the CoinbaseClient with Ed25519 format API keys

import os
import sys
from pathlib import Path
from datetime import datetime, timedelta, timezone
import pandas as pd
from dotenv import load_dotenv

# Find project root
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find project root")

PROJECT_ROOT = find_project_root(Path.cwd())
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# Load environment variables
load_dotenv(PROJECT_ROOT / ".env")

print(f"Project root: {PROJECT_ROOT}")
print(f"Python version: {sys.version.split()[0]}")

Project root: d:\data\development\crypto
Python version: 3.14.0


In [63]:
# Verify environment configuration
# Check that API credentials are loaded properly

def mask_value(value: str | None, visible: int = 4) -> str:
    """Mask sensitive values for display"""
    if not value:
        return "<MISSING>"
    if len(value) <= visible * 2:
        return "*" * len(value)
    return f"{value[:visible]}...{value[-visible:]}"

api_key = os.getenv("COINBASE_API_KEY")
api_secret = os.getenv("COINBASE_API_SECRET")
base_url = os.getenv("COINBASE_BASE_URL", "https://api.coinbase.com")

print("Environment Configuration:")
print(f"  COINBASE_API_KEY: {mask_value(api_key, 8)}")
print(f"  COINBASE_API_SECRET: {mask_value(api_secret, 6)}")
print(f"  COINBASE_BASE_URL: {base_url}")
print()
print(f"API Key Format: {'Ed25519 (organizations/...)' if api_key and 'organizations/' in api_key else 'Unknown'}")
print(f"API Secret Length: {len(api_secret) if api_secret else 0} characters")

Environment Configuration:
  COINBASE_API_KEY: organiza...96a0c63a
  COINBASE_API_SECRET: HK1Gzc...XzuA==
  COINBASE_BASE_URL: https://api.coinbase.com

API Key Format: Ed25519 (organizations/...)
API Secret Length: 88 characters


In [64]:
# Import CoinbaseClient and related models
from cryptoquant.collectors.coinbase_client import (
    CoinbaseClient,
    CoinbaseAPIError,
    CoinbaseAuthenticationError,
    CoinbaseRateLimitError,
)
from cryptoquant.collectors.models import CandleGranularity

# Initialize the client (will use credentials from environment)
client = CoinbaseClient()
print("✓ CoinbaseClient initialized successfully")

✓ CoinbaseClient initialized successfully


In [65]:
# Test 1: Get all products
# This tests authenticated API access with Ed25519 JWT

print("Test 1: Fetching all products...")
try:
    products = client.get_products(active_only=True)
    print(f"✓ Successfully retrieved {len(products)} active products")
    
    # Convert to DataFrame for analysis
    products_df = pd.DataFrame([p.model_dump() for p in products])
    print(f"\nProduct columns: {list(products_df.columns)}")
    
    # Display first 10 products
    display_cols = ["product_id", "base_currency_id", "quote_currency_id", "status", "trading_disabled"]
    display(products_df[display_cols].head(10))
    
except CoinbaseAuthenticationError as e:
    print(f"✗ Authentication failed: {e}")
except CoinbaseAPIError as e:
    print(f"✗ API error: {e}")
except Exception as e:
    print(f"✗ Unexpected error: {type(e).__name__}: {e}")

Test 1: Fetching all products...
✓ Successfully retrieved 912 active products

Product columns: ['product_id', 'base_currency_id', 'quote_currency_id', 'base_display_symbol', 'quote_display_symbol', 'status', 'trading_disabled', 'base_increment', 'quote_increment', 'base_min_size', 'base_max_size', 'quote_min_size', 'quote_max_size']


,product_id,base_currency_id,quote_currency_id,status,trading_disabled
0,BTC-USD,BTC,USD,online,False
1,BTC-USDC,BTC,USDC,online,False
2,ETH-USD,ETH,USD,online,False
3,ETH-USDC,ETH,USDC,online,False
4,XRP-USD,XRP,USD,online,False
5,XRP-USDC,XRP,USDC,online,False
6,USDT-USD,USDT,USD,online,False
7,SOL-USD,SOL,USD,online,False
8,SOL-USDC,SOL,USDC,online,False
9,ADA-USD,ADA,USD,online,False


In [66]:
# Test 2: Filter USD trading pairs
# Get only products quoted in USD and sort by volume

print("Test 2: Filtering USD trading pairs...")
try:
    usd_products = products_df[products_df["quote_currency_id"] == "USD"].copy()
    print(f"✓ Found {len(usd_products)} USD-quoted products")
    
    # Display top USD pairs
    display_cols = ["product_id", "base_currency_id", "status"]
    display(usd_products[display_cols].head(20))
    
except Exception as e:
    print(f"✗ Error filtering products: {e}")

Test 2: Filtering USD trading pairs...
✓ Found 397 USD-quoted products


,product_id,base_currency_id,status
0,BTC-USD,BTC,online
2,ETH-USD,ETH,online
4,XRP-USD,XRP,online
6,USDT-USD,USDT,online
7,SOL-USD,SOL,online
9,ADA-USD,ADA,online
12,HYPE-USD,HYPE,online
14,ZEC-USD,ZEC,online
16,BICO-USD,BICO,online
19,HFT-USD,HFT,online


In [67]:
# Test 3: Get historical candle data for BTC-USD
# Test different granularities and date ranges

product_id = "BTC-USD"
print(f"Test 3: Fetching candle data for {product_id}...")

try:
    # Get 7 days of daily candles
    candles = client.get_candles(
        product_id=product_id,
        granularity=CandleGranularity.ONE_DAY,
        days=7
    )
    print(f"✓ Successfully retrieved {len(candles)} daily candles")
    
    # Convert to DataFrame (start is already a datetime object)
    candles_df = pd.DataFrame([c.model_dump() for c in candles])
    
    # Display candle data
    print(f"\nDate range: {candles_df['start'].min()} to {candles_df['start'].max()}")
    display(candles_df)
    
except CoinbaseAPIError as e:
    print(f"✗ API error: {e}")
except Exception as e:
    print(f"✗ Unexpected error: {type(e).__name__}: {e}")

Test 3: Fetching candle data for BTC-USD...
✓ Successfully retrieved 7 daily candles

Date range: 2026-08-01 00:00:00+00:00 to 2026-08-07 00:00:00+00:00


,start,low,high,open,close,volume
0,2026-08-01 00:00:00+00:00,62209.81,63093,62826.69,62764.2,3060.55404241
1,2026-08-02 00:00:00+00:00,62743.87,63730.49,62764.2,63499.49,3283.94724012
2,2026-08-03 00:00:00+00:00,62210.14,64032.43,63499.49,63466.51,7086.70328594
3,2026-08-04 00:00:00+00:00,63250,64497.24,63466.51,64050.51,6227.77436334
4,2026-08-05 00:00:00+00:00,63814.8,64988.78,64050.51,64603.03,6932.34404967
5,2026-08-06 00:00:00+00:00,64087.41,64944.16,64603.02,64267.3,4520.43206155
6,2026-08-07 00:00:00+00:00,64099.35,65348.97,64267.3,64919.41,5291.27340732


In [68]:
# Test 4: Shorter time granularity (hourly candles)

product_id = "ETH-USD"
print(f"Test 4: Fetching hourly candles for {product_id}...")

try:
    # Get 24 hours of hourly candles
    candles = client.get_candles(
        product_id=product_id,
        granularity=CandleGranularity.ONE_HOUR,
        days=1
    )
    print(f"✓ Successfully retrieved {len(candles)} hourly candles")
    
    # Convert to DataFrame (start is already a datetime object)
    candles_df = pd.DataFrame([c.model_dump() for c in candles])
    
    # Show latest candles
    print(f"\nLatest {product_id} candles:")
    display(candles_df.tail(10))
    
except CoinbaseAPIError as e:
    print(f"✗ API error: {e}")
except Exception as e:
    print(f"✗ Unexpected error: {type(e).__name__}: {e}")

Test 4: Fetching hourly candles for ETH-USD...
✓ Successfully retrieved 24 hourly candles

Latest ETH-USD candles:


,start,low,high,open,close,volume
14,2026-08-07 10:00:00+00:00,1911.15,1914.8,1911.8,1911.43,1142.82469654
15,2026-08-07 11:00:00+00:00,1911.31,1919.73,1911.53,1914.89,1956.18166894
16,2026-08-07 12:00:00+00:00,1914.89,1942,1914.89,1930.99,9363.25212524
17,2026-08-07 13:00:00+00:00,1919.92,1934.12,1930.99,1922.06,5983.82784693
18,2026-08-07 14:00:00+00:00,1909,1925.4,1922.06,1912.19,7114.20889815
19,2026-08-07 15:00:00+00:00,1911.3,1919.69,1912.2,1916.59,3831.71129057
20,2026-08-07 16:00:00+00:00,1908,1923.92,1916.7,1908.09,4904.72160825
21,2026-08-07 17:00:00+00:00,1904.14,1911.79,1908.09,1911.01,4991.42204438
22,2026-08-07 18:00:00+00:00,1904.73,1914.5,1911.01,1908.52,2647.44485673
23,2026-08-07 19:00:00+00:00,1908.53,1919.4,1908.53,1917.34,6574.59078057


In [69]:
# Test 5: Test multiple products
# Verify client works for different trading pairs

test_products = ["BTC-USD", "ETH-USD", "SOL-USD", "AVAX-USD"]
print("Test 5: Testing multiple products...\n")

results = []
for product in test_products:
    try:
        candles = client.get_candles(
            product_id=product,
            granularity=CandleGranularity.ONE_DAY,
            days=1
        )
        if candles:
            latest = candles[-1]
            results.append({
                "product": product,
                "status": "✓ Success",
                "candles": len(candles),
                "latest_close": float(latest.close),
            })
            print(f"✓ {product}: {len(candles)} candles, latest close: ${float(latest.close):,.2f}")
    except Exception as e:
        results.append({
            "product": product,
            "status": f"✗ {type(e).__name__}",
            "candles": 0,
            "latest_close": 0,
        })
        print(f"✗ {product}: {e}")

# Summary
results_df = pd.DataFrame(results)
print(f"\nSummary:")
display(results_df)

Test 5: Testing multiple products...

✓ BTC-USD: 1 candles, latest close: $64,919.41
✓ ETH-USD: 1 candles, latest close: $1,917.34
✓ SOL-USD: 1 candles, latest close: $73.84
✓ AVAX-USD: 1 candles, latest close: $6.44

Summary:


,product,status,candles,latest_close
0,BTC-USD,✓ Success,1,64919.410
1,ETH-USD,✓ Success,1,1917.340
2,SOL-USD,✓ Success,1,73.840
3,AVAX-USD,✓ Success,1,6.439


In [70]:
# Test 6: Verify JWT token generation
# Test the low-level request method directly

print("Test 6: Testing low-level API request...")
try:
    # Make a direct request to best_bid_ask endpoint
    response = client._make_request(
        "GET",
        "/api/v3/brokerage/best_bid_ask",
        params={"product_ids": "BTC-USD"}
    )
    
    print("✓ Low-level request successful")
    
    # Display response
    if "pricebooks" in response:
        pricebooks = response["pricebooks"]
        if pricebooks:
            pb = pricebooks[0]
            print(f"\nBTC-USD Best Bid/Ask:")
            print(f"  Product ID: {pb.get('product_id')}")
            print(f"  Best Bid: ${pb.get('bids', [{}])[0].get('price', 'N/A') if pb.get('bids') else 'N/A'}")
            print(f"  Best Ask: ${pb.get('asks', [{}])[0].get('price', 'N/A') if pb.get('asks') else 'N/A'}")
            print(f"  Time: {pb.get('time')}")
    
except Exception as e:
    print(f"✗ Low-level request failed: {type(e).__name__}: {e}")

Test 6: Testing low-level API request...
✓ Low-level request successful

BTC-USD Best Bid/Ask:
  Product ID: BTC-USD
  Best Bid: $64919.41
  Best Ask: $64919.42
  Time: 2026-08-07T19:52:41.974518Z


In [71]:
# Test Summary
print("="*60)
print("COINBASE CLIENT TEST SUMMARY")
print("="*60)
print(f"Authentication: Ed25519 JWT")
print(f"API Key Format: {mask_value(api_key, 8)}")
print(f"Base URL: {base_url}")
print()
print("All tests completed. Check results above.")
print("If all tests passed, the CoinbaseClient is working correctly with Ed25519 authentication.")

COINBASE CLIENT TEST SUMMARY
Authentication: Ed25519 JWT
API Key Format: organiza...96a0c63a
Base URL: https://api.coinbase.com

All tests completed. Check results above.
If all tests passed, the CoinbaseClient is working correctly with Ed25519 authentication.
